In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

DATA_DIR = os.path.join(os.getcwd(), "Embedding", "6")

LABEL_TO_IDX = {'E': 0, 'D': 1, 'C': 2, 'B': 3, 'A': 4}
IDX_TO_LABEL = {v: k for k, v in LABEL_TO_IDX.items()}
N_CLASSES    = 5

In [2]:
# ── Load Data ─────────────────────────────────────────────────────────────────
# Input: output fasttext.ipynb (metadata sudah punya kolom labela)

questions_emb  = np.load(os.path.join(DATA_DIR, 'questions_emb.npy'))
answerkeys_emb = np.load(os.path.join(DATA_DIR, 'answerkeys_emb.npy'))
answers_emb    = np.load(os.path.join(DATA_DIR, 'answers_emb.npy'))
metadata       = pd.read_pickle(os.path.join(DATA_DIR, 'metadata.pkl'))
metadata       = metadata.reset_index(drop=True)
metadata['is_synthetic'] = False

# label_ae langsung dari kolom labela (sudah ada di metadata)
metadata['label_ae'] = metadata['labela'].map(LABEL_TO_IDX)

print("=== Hasil Load ===")
print(f"questions_emb  : {questions_emb.shape}")
print(f"answerkeys_emb : {answerkeys_emb.shape}")
print(f"answers_emb    : {answers_emb.shape}")
print(f"Metadata       : {len(metadata)} rows")
print(f"Kolom          : {list(metadata.columns)}")
print()
print("Distribusi label A-E:")
for idx in range(N_CLASSES):
    n = (metadata['label_ae'] == idx).sum()
    print(f"  {IDX_TO_LABEL[idx]}: {n}")

=== Hasil Load ===
questions_emb  : (17, 135, 300)
answerkeys_emb : (17, 90, 300)
answers_emb    : (1229, 85, 300)
Metadata       : 1229 rows
Kolom          : ['IDJwb', 'IDPSJ', 'grade', 'labela', 'psj_idx', 'is_synthetic', 'label_ae']

Distribusi label A-E:
  E: 323
  D: 155
  C: 189
  B: 214
  A: 348


In [3]:
# ── Analisis lubang (kelas A-E = 0 sampel per IDPSJ) ─────────────────────────
# CPI A-E hanya mengisi kelas yang benar-benar kosong di suatu IDPSJ.
# Jauh lebih sedikit dari CPI grade 1-10 (8 lubang vs 53 lubang).

pivot = (
    metadata.groupby(['IDPSJ', 'label_ae'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=range(N_CLASSES), fill_value=0)
)
pivot.columns = [IDX_TO_LABEL[c] for c in pivot.columns]

print("Distribusi A-E per IDPSJ sebelum CPI:")
print(pivot.to_string())

holes = [
    (psj, ae)
    for psj in pivot.index
    for ae in range(N_CLASSES)
    if pivot.loc[psj, IDX_TO_LABEL[ae]] == 0
]

print(f"\nTotal lubang A-E: {len(holes)}")
for psj, ae in holes:
    print(f"  IDPSJ {psj:>2}  kelas {IDX_TO_LABEL[ae]}")

Distribusi A-E per IDPSJ sebelum CPI:
        E   D   C   B   A
IDPSJ                    
1      10  10  25  14  42
2      16   5   5  10   5
3       5   5  10   6   7
4      10  16  10  10  20
5      28   5   5   5  10
6      30  10  10  10  11
7       9   8  10  10  20
8      18  11  10  10  63
9      23  10  10  12  48
10     13  10  10  41  40
11     65  10  13  10  22
12      5   5  21   6   8
13     10  10  10  20  10
14     11  10  10  20  12
15     25  10  10  10  10
16     16  10  10  10  10
17     29  10  10  10  10

Total lubang A-E: 0


In [ ]:
# ── Cross-Prompt Injection A-E ────────────────────────────────────────────────
# Untuk setiap lubang (IDPSJ_target, kelas_target):
#   1. target = ceil(max_kelas_IDPSJ / 2)
#   2. Donor : IDPSJ lain yang PUNYA kelas_target (label_ae sama)
#   3. Ambil 'target' answer embeddings dari donor (acak, dengan penggantian)
#   4. Pasangkan dengan question & answerkey milik IDPSJ_target
#   5. labela & label_ae tetap = kelas_target
#
# Perbedaan vs CPI grade 1-10:
#   - Donor dipilih berdasarkan kelas A-E, bukan integer grade
#   - Lebih sedikit injeksi (hanya kelas yang benar-benar kosong)
#   - Grade donor dipertahankan untuk referensi, label_ae adalah kelas yang diisi

RNG = np.random.default_rng(seed=42)

inj_answers = []
inj_meta    = []

for (tgt_psj, tgt_ae) in holes:
    tgt_label = IDX_TO_LABEL[tgt_ae]

    # Target = ceil(max kelas A-E di IDPSJ ini / 2)
    local_max = pivot.loc[tgt_psj].max()
    n_inject  = int(np.ceil(local_max / 2))

    # Donor: IDPSJ lain dengan kelas A-E yang sama
    donor_mask = (
        (metadata['IDPSJ'] != tgt_psj) &
        (metadata['label_ae'] == tgt_ae)
    )
    donor_idx = metadata.index[donor_mask].values

    if len(donor_idx) == 0:
        print(f"  [SKIP] IDPSJ={tgt_psj} kelas={tgt_label} — tidak ada donor!")
        continue

    print(f"  IDPSJ={tgt_psj} kelas={tgt_label}  target={n_inject}  donor={len(donor_idx)}")

    sampled = RNG.choice(donor_idx, size=n_inject,
                         replace=len(donor_idx) < n_inject)

    inj_answers.append(answers_emb[sampled])

    for k, src_idx in enumerate(sampled):
        inj_meta.append({
            'IDJwb'        : f'cpi_ae_{tgt_psj}_{tgt_label}_{k}',
            'IDPSJ'        : tgt_psj,
            'grade'        : metadata.loc[src_idx, 'grade'],   # grade donor (informatif)
            'labela'       : tgt_label,
            'label_ae'     : tgt_ae,
            'is_synthetic' : True,
        })

# ── Gabungkan ─────────────────────────────────────────────────────────────────
if inj_answers:
    final_answers_emb = np.vstack([answers_emb] + inj_answers)
    final_metadata    = pd.concat([metadata, pd.DataFrame(inj_meta)],
                                  ignore_index=True)
else:
    final_answers_emb = answers_emb.copy()
    final_metadata    = metadata.copy()

n_inj = len(final_metadata) - len(metadata)
print(f"\nSebelum CPI : {len(metadata)}")
print(f"Injeksi CPI : {n_inj}")
print(f"Total       : {len(final_metadata)}")

In [ ]:
# ── Verifikasi distribusi setelah CPI A-E ─────────────────────────────────────
pivot_after = (
    final_metadata.groupby(['IDPSJ', 'label_ae'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=range(N_CLASSES), fill_value=0)
)
pivot_after.columns = [IDX_TO_LABEL[c] for c in pivot_after.columns]

print("Distribusi A-E per IDPSJ setelah CPI:")
print(pivot_after.to_string())

remaining = [
    (psj, IDX_TO_LABEL[ae])
    for psj in pivot_after.index
    for ae in range(N_CLASSES)
    if pivot_after.loc[psj, IDX_TO_LABEL[ae]] == 0
]
print(f"\nSisa lubang A-E: {len(remaining)}")
if remaining:
    for psj, lbl in remaining:
        print(f"  IDPSJ={psj} kelas={lbl} — tidak ada donor di seluruh dataset")

# Bar chart sebelum vs sesudah
before = metadata['label_ae'].value_counts().sort_index()
after  = final_metadata['label_ae'].value_counts().sort_index()

x        = np.arange(N_CLASSES)
labels_x = [IDX_TO_LABEL[i] for i in range(N_CLASSES)]
w        = 0.4

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, [before.get(i, 0) for i in range(N_CLASSES)], w,
       label='Sebelum CPI', color='steelblue',  edgecolor='black')
ax.bar(x + w/2, [after.get(i,  0) for i in range(N_CLASSES)], w,
       label='Setelah CPI', color='darkorange', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(labels_x, fontsize=12)
ax.set_xlabel('Kelas')
ax.set_ylabel('Jumlah Sampel')
ax.set_title('Distribusi A-E Global Sebelum vs Setelah CPI A-E')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Simpan ────────────────────────────────────────────────────────────────────
# Output:
#   cpi_ae_answers_emb.npy  <- answers setelah CPI A-E (per sampel)
#   cpi_ae_metadata.pkl     <- metadata dengan kolom labela, label_ae, psj_idx
#
# questions_emb.npy & answerkeys_emb.npy TIDAK berubah — langsung dipakai
# oleh balancing_data_klasifikasi.ipynb dan training.

idpsj_sorted   = sorted(final_metadata['IDPSJ'].unique())
idpsj_to_idx   = {psj: i for i, psj in enumerate(idpsj_sorted)}
final_metadata = final_metadata.copy()
final_metadata['psj_idx'] = final_metadata['IDPSJ'].map(idpsj_to_idx)

np.save(os.path.join(DATA_DIR, 'cpi_ae_answers_emb.npy'), final_answers_emb)
final_metadata.to_pickle(os.path.join(DATA_DIR, 'cpi_ae_metadata.pkl'))

print("File tersimpan:")
print(f"  cpi_ae_answers_emb.npy   {final_answers_emb.shape}  (per sampel)")
print(f"  cpi_ae_metadata.pkl      {len(final_metadata)} rows")
print(f"  Kolom: {list(final_metadata.columns)}")
print()
print("File yang tetap dipakai (tidak berubah):")
print(f"  questions_emb.npy    {questions_emb.shape}  (per IDPSJ)")
print(f"  answerkeys_emb.npy   {answerkeys_emb.shape}  (per IDPSJ)")
print()
# Verifikasi psj_idx
for psj in sorted(final_metadata['IDPSJ'].unique()):
    idx_vals = final_metadata.loc[final_metadata['IDPSJ'] == psj, 'psj_idx'].unique()
    assert len(idx_vals) == 1
print("Verifikasi psj_idx: OK")